# AF2Rank Pipeline

Score protein-complex structures on a Colab GPU. The bundled 1BRS example is ready to run; upload a FASTA and models to try your own target. Run the sections in order.


## Setup

Run this section once per Colab session. It prepares inference and downloads the required AlphaFold weight (about 373 MB on the first run).


In [ ]:
#@title 1. Find project and check GPU { display-mode: "form" }
REPOSITORY_URL = "https://github.com/sahinmtaylan/af2rank-pipeline.git"

from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys

workspace = Path("/content") if Path("/content").is_dir() else Path.cwd()
possible_repos = [workspace / "af2rank-pipeline", workspace]
possible_repos += [p.parent for p in workspace.glob("*/pyproject.toml")]
possible_repos += [p.parent for p in workspace.glob("*/*/pyproject.toml")]
repo = next((p for p in possible_repos
             if (p / "pyproject.toml").is_file()
             and (p / "uv.lock").is_file()
             and (p / "examples/1brs/model.pdb").is_file()), None)
if repo is None:
    repo = workspace / "af2rank-pipeline"
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(repo)], check=True)
repo = repo.resolve()

import jax
if not any(device.platform == "gpu" for device in jax.devices()):
    raise RuntimeError("JAX cannot see a GPU. In Colab, select Runtime → Change runtime type → GPU.")
print("Project:", repo)
print("Colab JAX:", jax.__version__, "·", jax.devices())


In [ ]:
#@title 2. Prepare inference environment { display-mode: "form" }
# The repository's tested inference stack pins JAX 0.4.20 and Python 3.10.
# Current Colab images can provide a newer JAX; reuse the host CUDA driver
# while keeping these pinned Python packages in a cached project environment.
pipeline_env = {**os.environ, "MPLBACKEND": "Agg"}
pipeline_python = repo / ".venv/bin/python"
pipeline_cli = repo / ".venv/bin/af2rank-pipeline"

def check_inference_environment():
    return subprocess.run([
        str(pipeline_python), "-c",
        "import sys, jax, colabdesign; print('Python:', sys.version.split()[0]); "
        "print('JAX:', jax.__version__, 'devices:', jax.devices()); "
        "assert any(d.platform == 'gpu' for d in jax.devices()), 'JAX does not see a GPU'",
    ], capture_output=True, text=True, cwd=repo, env=pipeline_env)

probe = (check_inference_environment()
         if pipeline_python.is_file() and pipeline_cli.is_file() else None)
if probe is None or probe.returncode:
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.12.20"], check=True)
    uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("uv was installed but is not on PATH")
    subprocess.run([uv, "python", "install", "3.10"], check=True, cwd=repo)
    subprocess.run([uv, "sync", "--locked", "--extra", "inference", "--extra", "dockq"], check=True, cwd=repo)
    subprocess.run([
        uv, "pip", "install", "--python", str(pipeline_python),
        "--constraint", str(repo / "env/cuda12-constraints.txt"),
        "--find-links", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
        "jax[cuda12_pip]==0.4.20",
    ], check=True, cwd=repo)
    probe = check_inference_environment()
if probe.returncode:
    raise RuntimeError(f"Inference setup failed:\n{probe.stdout}{probe.stderr}")
print(probe.stdout.strip())

# These packages render results in the Colab kernel; they do not change the
# private inference environment or Colab's preinstalled JAX/CUDA stack.
viewer_packages = {
    "py3Dmol": "py3Dmol==2.5.5",
    "ipywidgets": "ipywidgets>=8,<9",
    "pandas": "pandas>=2,<4",
    "matplotlib": "matplotlib>=3.8,<4",
}
missing = [pin for module, pin in viewer_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
print("Inference and result viewer ready")


In [ ]:
#@title 3. Download model weight { display-mode: "form" }
params = workspace / "af2rank-weights" / "params"
subprocess.run([sys.executable, str(repo / "env/download_default_params.py"), str(params)],
               check=True, cwd=repo)
model_weight = params / "params_model_2_ptm.npz"
print("Model weight ready:", model_weight.name)


## Input

The bundled 1BRS FASTA and structure are selected by default. Upload files below to replace either one. Custom targets need one FASTA with a sequence for each chain and 1–5 PDB or mmCIF structures.


In [ ]:
#@title Choose FASTA and model files { display-mode: "form" }
import html
import ipywidgets as widgets
from IPython.display import display

fasta_upload = widgets.FileUpload(accept=".fa,.fasta,.faa", multiple=False,
                                  description="Upload FASTA")
model_upload = widgets.FileUpload(accept=".pdb,.pdb.gz,.cif,.mmcif", multiple=True,
                                  description="Upload models")
input_status = widgets.HTML()

def uploaded_files(widget):
    value = widget.value
    if isinstance(value, dict):  # ipywidgets 7
        return [{"name": name, **entry} for name, entry in value.items()]
    return list(value)  # ipywidgets 8

def refresh_input_status(change=None):
    fastas, models = uploaded_files(fasta_upload), uploaded_files(model_upload)
    fasta_name = fastas[0]["name"] if fastas else "bundled 1BRS FASTA"
    model_names = ", ".join(item["name"] for item in models) if models else "bundled 1BRS model"
    input_status.value = ("<div style='padding:10px 12px;background:#eef6f5;border-radius:8px'>"
                          f"<b>FASTA:</b> {html.escape(fasta_name)} &nbsp;·&nbsp; "
                          f"<b>Models:</b> {html.escape(model_names)}</div>")

fasta_upload.observe(refresh_input_status, names="value")
model_upload.observe(refresh_input_status, names="value")
refresh_input_status()
display(widgets.VBox([
    widgets.HBox([widgets.VBox([widgets.HTML("<b>Target FASTA</b>"), fasta_upload]),
                  widgets.VBox([widgets.HTML("<b>Structures</b>"), model_upload])]),
    input_status,
]))


## Run


In [ ]:
#@title Score models { display-mode: "form" }
from datetime import datetime, timezone
import re
import tempfile

fastas = uploaded_files(fasta_upload)
uploaded_models = uploaded_files(model_upload)
if len(fastas) > 1 or len(uploaded_models) > 5:
    raise ValueError("Upload one FASTA and at most five structures")

target = "1brs"
if fastas:
    target = re.sub(r"[^A-Za-z0-9_-]+", "_", Path(fastas[0]["name"]).stem).strip("_-") or "target"

fasta = repo / "examples/1brs/target.fasta"
models = repo / "examples/1brs/model.pdb"
if fastas or uploaded_models:
    input_dir = Path(tempfile.mkdtemp(prefix="af2rank-input-", dir=workspace))
    if fastas:
        name = Path(fastas[0]["name"]).name
        if not name.lower().endswith((".fa", ".fasta", ".faa")):
            raise ValueError("FASTA must have a .fa, .fasta, or .faa extension")
        fasta = input_dir / name
        fasta.write_bytes(bytes(fastas[0]["content"]))
    if uploaded_models:
        models = input_dir / "models"
        models.mkdir()
        seen_names = set()
        for item in uploaded_models:
            name = Path(item["name"]).name
            if not name.lower().endswith((".pdb", ".pdb.gz", ".cif", ".mmcif")):
                raise ValueError(f"Unsupported model file: {name}")
            if name in seen_names:
                raise ValueError(f"Duplicate model filename: {name}")
            seen_names.add(name)
            (models / name).write_bytes(bytes(item["content"]))

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
out = workspace / "af2rank-results" / f"{target}-{stamp}"
print(f"Scoring {len(uploaded_models) or 1} model(s) for {target}…", flush=True)
subprocess.run([
    str(pipeline_cli), "run-target",
    "--target", target,
    "--fasta", str(fasta),
    "--models", str(models),
    "--out", str(out),
    "--params", str(params),
    "--workers", "1",
], check=True, cwd=repo, env=pipeline_env)
print("Done:", out)


## Results


In [ ]:
#@title Score overview { display-mode: "form" }
import pandas as pd
from IPython.display import HTML, display

final_csv = out / "manifests/final_models.csv"
if not final_csv.is_file():
    raise FileNotFoundError(f"Results table not found: {final_csv}")
results = pd.read_csv(final_csv)
if results.empty:
    raise RuntimeError("No models were scored; inspect the run output")

overview_columns = {
    "model_id": "Model", "plddt": "pLDDT", "ptm": "pTM",
    "i_ptm": "interface pTM", "actifptm": "actifpTM",
    "dockq_io": "DockQ input→output", "pae_contacts_mean": "Contact PAE (Å)",
}
overview = results[[key for key in overview_columns if key in results.columns]].rename(
    columns=overview_columns).copy()
for name in overview.columns:
    if name != "Model":
        overview[name] = pd.to_numeric(overview[name], errors="coerce").round(3)
display(overview)
display(HTML("<small>DockQ compares the cleaned input with AF2Rank's output; "
             "it is not accuracy against an experimental structure. "
             "The ZIP includes every metric and structure.</small>"))
model_picker = widgets.Dropdown(
    options=[(str(row["model_id"]), i) for i, row in results.iterrows()],
    description="Inspect:", layout=widgets.Layout(width="360px"))
display(model_picker)


In [ ]:
#@title Input and rescored structures { display-mode: "form" }
import py3Dmol
from IPython.display import clear_output

color_picker = widgets.ToggleButtons(options=["Chains", "Output pLDDT"], description="Color:")
structure_output = widgets.Output()
chain_colors = ["#0f9d8a", "#e5a83b", "#8068c5", "#d96d62", "#3b88bd"]

def result_path(value):
    path = Path(str(value))
    if path.is_file():
        return path
    # The CSV records the original absolute path; resolve it after moving a run.
    parts = path.parts
    if out.name in parts:
        return out.joinpath(*parts[parts.index(out.name) + 1:])
    for directory in ("cleaned", "af2rank", "additional_metrics", "manifests"):
        if directory in parts:
            return out.joinpath(*parts[parts.index(directory):])
    return out / path

def draw_structures(change=None):
    row = results.iloc[model_picker.value]
    paths = [result_path(row[name]) for name in ("input_pdb", "output_pdb")]
    with structure_output:
        clear_output(wait=True)
        if not all(path.is_file() for path in paths):
            print("Missing structure:", *paths, sep="\n")
            return
        view = py3Dmol.view(width=900, height=420, viewergrid=(1, 2))
        for panel, path in enumerate(paths):
            position = (0, panel)
            pdb_text = path.read_text()
            view.addModel(pdb_text, "pdb", viewer=position)
            if panel == 1 and color_picker.value == "Output pLDDT":
                view.setStyle({"cartoon": {"colorscheme": {
                    "prop": "b", "gradient": "roygb", "min": 50, "max": 90,
                }}}, viewer=position)
            else:
                view.setStyle({"cartoon": {"color": "#9ca3af"}}, viewer=position)
                chains = list(dict.fromkeys(line[21] for line in pdb_text.splitlines()
                                            if line.startswith("ATOM") and len(line) > 21))
                for chain, color in zip(chains, chain_colors):
                    view.setStyle({"chain": chain}, {"cartoon": {"color": color}}, viewer=position)
            view.zoomTo(viewer=position)
        view.show()

model_picker.observe(draw_structures, names="value")
color_picker.observe(draw_structures, names="value")
display(HTML("<b>Left:</b> cleaned input &nbsp;&nbsp; <b>Right:</b> AF2Rank output"))
display(color_picker, structure_output)
draw_structures()


In [ ]:
#@title Predicted aligned error { display-mode: "form" }
import gzip
import json
import matplotlib.pyplot as plt
import numpy as np

pae_output = widgets.Output()
target_spec = json.loads((out / "target_spec.json").read_text())
chain_lengths = [item["length"] for item in target_spec["chains"]]

def draw_pae(change=None):
    row = results.iloc[model_picker.value]
    path = result_path(row["pae_json"])
    with pae_output:
        clear_output(wait=True)
        with gzip.open(path, "rt") as source:
            matrix = np.asarray(json.load(source)["pae"])
        fig, ax = plt.subplots(figsize=(6.4, 5.4))
        image = ax.imshow(matrix, cmap="magma_r", vmin=0, vmax=30,
                          origin="upper", interpolation="nearest")
        boundaries = np.cumsum(chain_lengths)[:-1]
        for boundary in boundaries:
            ax.axhline(boundary - 0.5, color="white", lw=1)
            ax.axvline(boundary - 0.5, color="white", lw=1)
        ticks = [sum(chain_lengths[:i]) + length / 2 - 0.5
                 for i, length in enumerate(chain_lengths)]
        labels = [item["canonical_id"] for item in target_spec["chains"]]
        ax.set_xticks(ticks, labels)
        ax.set_yticks(ticks, labels)
        ax.set_xlabel("Residue / chain")
        ax.set_ylabel("Residue / chain")
        ax.set_title(f"PAE · {row['model_id']}")
        fig.colorbar(image, ax=ax, label="Predicted aligned error (Å)")
        fig.tight_layout()
        display(fig)
        plt.close(fig)

model_picker.observe(draw_pae, names="value")
display(pae_output)
draw_pae()


In [ ]:
#@title AF2Rank and interface details { display-mode: "form" }
detail_output = widgets.Output()
dockq_table = pd.read_csv(out / "additional_metrics/dockq_metrics.csv")
pae_table = pd.read_csv(out / "additional_metrics/pae_summary.csv")

metric_names = {
    "plddt": "Mean pLDDT", "ptm": "pTM", "i_ptm": "Interface pTM",
    "actifptm": "actifpTM", "actifptm_prob": "actifpTM probability",
    "actifptm_binary": "actifpTM binary", "pae": "Mean PAE (Å)",
    "tm_i": "TM-score input", "tm_o": "TM-score output",
    "tm_io": "TM-score input→output", "composite": "Composite score",
    "pae_contacts_mean": "Contact PAE mean (Å)",
    "pae_contacts_min": "Contact PAE min (Å)",
    "pae_contacts_max": "Contact PAE max (Å)",
    "pae_contacts_count": "PAE contacts", "missing_residue_count": "Missing residues",
    "skipped_mismatch_count": "Skipped mismatches",
}

def show_details(change=None):
    row = results.iloc[model_picker.value]
    model_id = str(row["model_id"])
    with detail_output:
        clear_output(wait=True)
        per_chain = [name for name in results.columns
                     if name.startswith(("chain_ptm_", "actifptm_", "iptm_"))
                     and name not in ("actifptm_prob", "actifptm_binary")]
        names = [name for name in (*metric_names, *per_chain) if name in results.columns]
        values = pd.DataFrame({"Metric": [metric_names.get(name, name) for name in names],
                               "Value": [row[name] for name in names]})
        values = values[values["Value"].notna() & (values["Value"].astype(str) != "N/A")]
        display(HTML("<h4>Confidence and AF2Rank scores</h4>"))
        display(values.reset_index(drop=True))
        matches = dockq_table["model_id"].astype(str) == model_id
        if "run_name" in dockq_table:
            matches |= dockq_table["run_name"].astype(str) == model_id
        interface_rows = dockq_table[matches]
        dockq_columns = [name for name in
                         ("interface", "DockQ", "F1", "iRMSD", "LRMSD", "fnat", "clashes")
                         if name in interface_rows.columns]
        display(HTML("<h4>Input–output interface comparison</h4>"))
        display(interface_rows[dockq_columns].round(3).reset_index(drop=True))
        directional = pae_table[pae_table["model_id"].astype(str) == model_id]
        pae_columns = [name for name in
                       ("interface", "direction", "pae_count", "pae_mean", "pae_min", "pae_max")
                       if name in directional.columns]
        display(HTML("<h4>Directional interface PAE</h4>"))
        display(directional[pae_columns].round(3).reset_index(drop=True))

model_picker.observe(show_details, names="value")
display(detail_output)
show_details()


In [ ]:
#@title Download complete results { display-mode: "form" }
archive = shutil.make_archive(str(out), "zip", root_dir=out)
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(archive))
else:
    files.download(archive)


The full results archive contains scored PDBs, PAE JSON, per-interface measurements, and the complete CSV. TM-score and its composite score remain empty unless the optional TMscore executable is installed. AlphaFold parameters are supplied under [CC BY 4.0](https://github.com/google-deepmind/alphafold#model-parameters).
